In [1]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# Percorsi
first_step_path = Path("dataset_big/first_step.gpkg")
chain_path = Path("dataset_big/chain_with_cost.gpkg")

# -------------------------------
# 1. Caricamento dati
# -------------------------------
# Colleague's layers
filling_first = gpd.read_file(first_step_path, layer="filling_points")
# User's layers
filling_chain = gpd.read_file(chain_path, layer="filling")
resell_chain = gpd.read_file(chain_path, layer="resell")

# -------------------------------
# 2. Aggiornamento filling_points con markup
# -------------------------------
# Calcola markup dai tuoi dati (cost_fil_kg_out - cost_fil_kg_in)
markup_map = filling_chain.set_index("id_res&fil")[["cost_fil_kg_out", "cost_fil_kg_in"]].copy()
markup_map["cost_fil_plants"] = markup_map["cost_fil_kg_out"] - markup_map["cost_fil_kg_in"]

# Unisci al filling del collega (usando l'ID comune)
filling_first = filling_first.merge(
    markup_map[["cost_fil_plants"]],
    left_on="id_res&fil",
    right_index=True,
    how="left"
)
# Gestisci i NaN (dovrebbero essere 0 se tutti i filling sono mappati)
filling_first["cost_fil_plants"] = filling_first["cost_fil_plants"].fillna(0.0)

# Aggiorna LPG_price: nuovo = vecchio + markup
filling_first["LPG_price"] = filling_first["LPG_price"] + filling_first["cost_fil_plants"]

# (Opzionale) rinominiamo la colonna per chiarezza, ma la lasciamo come LPG_price
# Se vuoi conservare il vecchio LPG_price come cost_fil_kg_in, puoi rinominarlo.
# filling_first.rename(columns={"LPG_price": "cost_fil_kg_in_original"}, inplace=False)

# -------------------------------
# 3. Creazione layer resell unificato
# -------------------------------
# Seleziona le colonne di base dal tuo resell_chain
resell_new = resell_chain[["id_res&fil", "geometry", "filling_reference",
                           "cost_res_kg_in", "cost_res_kg_out"]].copy()
resell_new.rename(columns={"id_res&fil": "id_resell"}, inplace=True)

# Prepara un dataframe con le colonne del filling che vuoi copiare (quelle nell'immagine)
cols_to_copy = [
    "cost_transport_from_storage", "LPG_price", "cost_source", "cost_import_sea",
    "cost_sts", "cost_pre_bottling", "cost_import_land", "cost_border_wait",
    "cost_ferry", "cost_transport_to_storage", "cost_storage", "cost_storage_second",
    "cost_rebalancing"
]
# Aggiungi anche l'ID del filling per il merge
filling_for_merge = filling_first[["id_res&fil"] + cols_to_copy].copy()
filling_for_merge.rename(columns={"id_res&fil": "filling_id"}, inplace=True)

# Merge tra resell e filling (usando filling_reference -> filling_id)
resell_new = resell_new.merge(
    filling_for_merge,
    left_on="filling_reference",
    right_on="filling_id",
    how="left"
)
# Controllo: se qualche merge fallisce, dai un warning
if resell_new["filling_id"].isna().any():
    print(f"Warning: {resell_new['filling_id'].isna().sum()} reseller senza filling corrispondente")

# Calcola cost_res.shop e cost_truck (usando i tuoi valori)
resell_new["cost_res_shop"] = resell_new["cost_res_kg_out"] - resell_new["cost_res_kg_in"]
resell_new["cost_truck"] = resell_new["cost_res_kg_in"] - resell_new["LPG_price"]   # LPG_price qui è quello del filling (già aggiornato)

# Calcola il nuovo LPG_price per il reseller = LPG_price del filling + cost_truck + cost_res_shop
resell_new["LPG_price"] = resell_new["LPG_price"] + resell_new["cost_truck"] + resell_new["cost_res_shop"]

# (Facoltativo) Puoi aggiungere anche altre colonne di dettaglio, ad esempio:
# resell_new["cost_res_kg_in_reconstructed"] = resell_new["LPG_price"] - resell_new["cost_res_shop"] - resell_new["cost_truck"]

# -------------------------------
# 4. Salvataggio in first_step.gpkg (sovrascrive i layer esistenti)
# -------------------------------
# Nota: usiamo mode='w' per sovrascrivere i layer specificati.
# Attenzione: questo elimina gli altri layer? No, gpkg permette di scrivere layer singoli.
# Usiamo un contesto per evitare conflitti.

# Scrivi il filling aggiornato (sostituisce il layer esistente)
filling_first.to_file(first_step_path, layer="filling_points", driver="GPKG", mode="w")

# Scrivi il nuovo resell (aggiunge il layer, se esiste lo sovrascrive)
resell_new.to_file(first_step_path, layer="resell", driver="GPKG", mode="a")

print("Unificazione completata. I layer 'filling_points' e 'resell' sono stati aggiornati in first_step.gpkg.")

Unificazione completata. I layer 'filling_points' e 'resell' sono stati aggiornati in first_step.gpkg.
